In [1]:
import os
import sys
import math
import shutil
from pathlib import Path
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
# !gcloud auth application-default login

In [3]:
# =============================================================
# CELL 5 — HÀM TIỆN ÍCH DÙNG CHUNG
# =============================================================
_JUNK = ('', 'null', 'NULL', 'N/A', 'n/a', 'None', 'none', 'NaN', 'nan')


def clean_string_columns(df):
    """Trim + chuẩn hoá giá trị rác cho tất cả cột string."""
    for col_name, dtype in df.dtypes:
        if dtype == 'string':
            df = df.withColumn(col_name, F.trim(F.col(col_name)))
            df = df.withColumn(
                col_name,
                F.when(F.col(col_name).isin(*_JUNK), F.lit(None))
                 .otherwise(F.col(col_name))
            )
    return df


def safe_cast(df, col_name: str, target_type: str):
    """Cast an toàn: chuẩn hoá rác → NULL trước, rồi mới cast kiểu."""
    raw = F.trim(F.col(col_name).cast('string'))
    return df.withColumn(
        col_name,
        F.when(raw.isin(*_JUNK), F.lit(None))
         .otherwise(F.col(col_name))
         .cast(target_type)
    )


print('✅ clean_string_columns() và safe_cast() đã sẵn sàng.')


✅ clean_string_columns() và safe_cast() đã sẵn sàng.


In [4]:
# =============================================================
# CELL 6 — HÀM TIỀN XỬ LÝ REVIEW & META
# =============================================================

def process_review_silver(df, cat_name):
    df = df.withColumn("source_category", F.lit(cat_name))

    # 1. Quét rác toàn cục
    df = clean_string_columns(df)

    # 2. Ép kiểu an toàn
    raw_ts = F.trim(F.col("timestamp_raw").cast("string"))
    df = df.withColumnRenamed("timestamp", "timestamp_raw") \
           .withColumn("timestamp",
               F.to_timestamp(
                   F.when(raw_ts.isin("", "null"), F.lit(None))
                    .otherwise(F.col("timestamp_raw")) / 1000
               ))

    raw_rating = F.trim(F.col("rating").cast("string"))
    df = df.withColumn("rating",
        F.when(raw_rating.isin("", "null"), F.lit(None))
         .otherwise(F.col("rating")).cast("byte"))

    raw_vp = F.trim(F.col("verified_purchase").cast("string"))
    df = df.withColumn("verified_purchase",
        F.when(raw_vp.isin("", "null"), F.lit(None))
         .otherwise(F.col("verified_purchase")).cast("boolean"))

    raw_hv = F.trim(F.col("helpful_vote").cast("string"))
    df = df.withColumn("helpful_vote",
        F.when(raw_hv.isin("", "null"), F.lit(None))
         .otherwise(F.col("helpful_vote")).cast("int"))

    # 3. Lọc bỏ các dòng thiếu dữ liệu cốt lõi
    review_drop_cols = ["user_id", "parent_asin", "rating", "timestamp",
                        "verified_purchase", "helpful_vote"]
    df = df.dropna(subset=[c for c in review_drop_cols if c in df.columns])

    # 4. Làm sạch văn bản bằng Regex
    df = df.withColumn("text", F.regexp_replace(F.col("text").cast("string"), r"<[^>]*>", " ")) \
           .withColumn("text", F.regexp_replace(F.col("text"), r"https?://\S+|www\.\S+", "")) \
           .withColumn("text", F.regexp_replace(F.col("text"), r"[^a-zA-Z0-9\s]", "")) \
           .withColumn("text", F.trim(F.regexp_replace(F.col("text"), r"\s+", " ")))

    # 5. Loại trùng lặp
    window_dedup = Window.partitionBy("user_id", "parent_asin", "timestamp_raw") \
                         .orderBy(F.col("helpful_vote").desc_nulls_last())
    df = df.withColumn("rn", F.row_number().over(window_dedup)) \
           .filter(F.col("rn") == 1).drop("rn")

    df = df.withColumn("processed_at", F.current_timestamp())
    core_cols = ["user_id", "parent_asin", "rating", "text", "timestamp",
                 "timestamp_raw", "verified_purchase", "helpful_vote",
                 "source_category", "processed_at"]
    return df.select(*[c for c in core_cols if c in df.columns])


def process_meta_silver(df, cat_name):
    df = df.withColumn("source_category", F.lit(cat_name))

    # 1. Quét rác toàn cục
    df = clean_string_columns(df)

    # 2. Xóa các cột không cần thiết
    _COLS_TO_DROP = ["author", "subtitle", "bought_together", "details", "videos"]
    existing_drop = [c for c in _COLS_TO_DROP if c in df.columns]
    if existing_drop:
        df = df.drop(*existing_drop)

    # 3. Ép kiểu an toàn
    if "average_rating" in df.columns:
        is_empty = F.trim(F.col("average_rating").cast("string")).isin("", "null", "N/A")
        df = df.withColumn("average_rating",
            F.when(is_empty, F.lit(None)).otherwise(F.col("average_rating")).cast("float"))

    if "rating_number" in df.columns:
        is_empty = F.trim(F.col("rating_number").cast("string")).isin("", "null", "N/A")
        df = df.withColumn("rating_number",
            F.when(is_empty, F.lit(None)).otherwise(F.col("rating_number")).cast("int"))

    if "price" in df.columns:
        extracted = F.regexp_extract(F.col("price").cast("string"), r"(\d+\.\d+|\d+)", 1)
        df = df.withColumn("price",
            F.when(extracted == "", F.lit(None)).otherwise(extracted).cast("float"))

    # 4. Xử lý cột images: lấy URL đầu tiên không null / không rỗng
    #    Dùng F.get(..., 0) thay vì [...][0] để tránh INVALID_ARRAY_INDEX khi mảng rỗng
    if "images" in df.columns:
        images_dtype = dict(df.dtypes).get("images", "")
        if "struct" in images_dtype:
            # Array<struct<large, hi_res, thumb, ...>>
            df = df.withColumn(
                "image_url",
                F.get(
                    F.filter(
                        F.transform(
                            F.col("images"),
                            lambda img: F.coalesce(img["large"], img["hi_res"], img["thumb"])
                        ),
                        lambda url: url.isNotNull() & (F.trim(url) != F.lit(""))
                    ), 0)
            ).drop("images")
        else:
            # Array<string>
            df = df.withColumn(
                "image_url",
                F.get(
                    F.filter(
                        F.col("images"),
                        lambda url: url.isNotNull() & (F.trim(url) != F.lit(""))
                    ), 0)
            ).drop("images")

    # 5. Lọc bỏ dòng thiếu ID sản phẩm
    df = df.dropna(subset=["parent_asin"])

    # 6. Chuẩn hóa Array rỗng
    for c in ["categories", "features", "description"]:
        if c in df.columns:
            df = df.withColumn(c, F.when(F.col(c).isNull(), F.array()).otherwise(F.col(c)))

    # 7. Loại trùng lặp
    window_meta = Window.partitionBy("parent_asin").orderBy(F.col("parent_asin"))
    df = df.withColumn("rn", F.row_number().over(window_meta)) \
           .filter(F.col("rn") == 1).drop("rn")

    df = df.withColumn("processed_at", F.current_timestamp())
    return df


print('✅ process_review_silver() và process_meta_silver() đã sẵn sàng.')


✅ process_review_silver() và process_meta_silver() đã sẵn sàng.


In [ ]:
def get_credentials_location():
    """Get credentials location based on OS (Windows vs Linux)"""
    home = Path.home()
    if sys.platform == "win32":
        return home / "AppData" / "Roaming" / "gcloud" / "application_default_credentials.json"
    else:
        # Linux/Mac
        return home / ".config" / "gcloud" / "application_default_credentials.json"

def resolve_credentials_location():
    """Resolve credentials path from env var first, then common defaults."""
    candidates = []

    env_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
    if env_path:
        candidates.append(Path(env_path).expanduser())

    candidates.append(get_credentials_location())

    # Helpful local fallbacks when running from notebook folders.
    cwd = Path.cwd()
    candidates.append(cwd / "application_default_credentials.json")
    candidates.append(cwd / "service-account.json")

    for candidate in candidates:
        if Path(candidate).exists():
            return str(candidate), [str(p) for p in candidates]

    return str(candidates[0]), [str(p) for p in candidates]

def get_gcs_jar_path():
    """Get or locate GCS connector jar"""
    jar_filename = "gcs-connector-hadoop3-latest.jar"
    # __file__ is not available in notebook cells; fall back to cwd.
    current_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    jar_path = current_dir / jar_filename
    
    if jar_path.exists():
        return str(jar_path)
    
    home = Path.home()
    jar_in_home = home / jar_filename
    if jar_in_home.exists():
        return str(jar_in_home)
    
    # Try temp location on VM
    if not sys.platform.startswith("win"):
        temp_jars = Path("/tmp") / jar_filename
        if temp_jars.exists():
            return str(temp_jars)
    
    return str(jar_path)  # Return expected path even if not found yet

def create_spark_session(
    driver_memory_gb=8,
    executor_memory_gb=8,
    reserve_cores=1,
    credentials_location=None,
):
    # Get paths dynamically (works on local Windows and VM Linux)
    if credentials_location is None:
        credentials_location, searched_paths = resolve_credentials_location()
    else:
        credentials_location = str(Path(credentials_location).expanduser())
        searched_paths = [credentials_location]
    gcs_jar_path = str(get_gcs_jar_path())

    if not Path(credentials_location).exists():
        searched_str = "\n - ".join(searched_paths)
        raise FileNotFoundError(
            f"Credentials file not found. Tried:\n - {searched_str}\n\n"
            "Fix options:\n"
            "1) Run: gcloud auth application-default login\n"
            "2) Or set env var GOOGLE_APPLICATION_CREDENTIALS to your service-account JSON path"
        )

    if not Path(gcs_jar_path).exists():
        raise FileNotFoundError(f"GCS connector jar not found: {gcs_jar_path}")

    # Resource tuning for local Spark (change these numbers if needed).
    total_cores = os.cpu_count() or 4
    spark_cores = max(1, total_cores - reserve_cores)
    default_parallelism = max(8, spark_cores * 2)
    shuffle_partitions = max(16, spark_cores * 4)
    jar_classpath = gcs_jar_path

    # Recreate Spark context so new config is applied on rerun.
    active_sc = SparkContext._active_spark_context
    if active_sc is not None:
        active_sc.stop()

    conf = (
        SparkConf()
            .setMaster(f"local[{spark_cores}]")
            .setAppName("bronze-to-silver")
            .set("spark.driver.memory", f"{driver_memory_gb}g")
            .set("spark.executor.memory", f"{executor_memory_gb}g")
            .set("spark.default.parallelism", str(default_parallelism))
            .set("spark.sql.shuffle.partitions", str(shuffle_partitions))
            # Use classpath instead of spark.jars to avoid Windows winutils/chmod issue.
            .set("spark.driver.extraClassPath", jar_classpath)
            .set("spark.executor.extraClassPath", jar_classpath)
            .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)
    )

    sc = SparkContext(conf=conf)
    hadoop_conf = sc._jsc.hadoopConfiguration()

    hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)

    spark = SparkSession.builder \
        .config(conf=sc.getConf()) \
        .getOrCreate()
    spark.conf.set("spark.sql.caseSensitive", "true")
    print(f"Spark cores: {spark_cores}/{total_cores}")
    print(f"Driver memory: {driver_memory_gb}g, Executor memory: {executor_memory_gb}g")
    print(f"defaultParallelism={default_parallelism}, shufflePartitions={shuffle_partitions}")
    return spark, sc

def _estimate_output_files_from_parquet_size(
    parquet_path,
    sc,
    target_parquet_file_mb=128,
    min_files=1,
    max_files=200,
):
    """
    Estimate output parquet file count from source parquet dataset size.
    
    output_files ~= source_parquet_size / target_parquet_file_mb
    """
    path = sc._jvm.org.apache.hadoop.fs.Path(parquet_path)
    fs = path.getFileSystem(sc._jsc.hadoopConfiguration())
    parquet_size_bytes = fs.getContentSummary(path).getLength()
    target_bytes = target_parquet_file_mb * 1024 * 1024

    estimated_files = math.ceil(parquet_size_bytes / target_bytes)
    output_files = max(min_files, min(max_files, estimated_files))
    return output_files, parquet_size_bytes

def _gcs_path_exists(gcs_path, sc):
    path = sc._jvm.org.apache.hadoop.fs.Path(gcs_path)
    fs = path.getFileSystem(sc._jsc.hadoopConfiguration())
    return fs.exists(path)

def apply_etl_transformations(df, category, data_type="review"):
    """
    Apply ETL transformations to dataframe.
    
    Parameters:
    -----------
    df : pyspark.sql.DataFrame
        Input dataframe from bronze zone
    category : str
        Product category name
    data_type : str
        Type of data: 'review' or 'metadata'
    
    Returns:
    --------
    pyspark.sql.DataFrame
        Transformed dataframe
    """
    print(f"Applying ETL transformations for {data_type} category: {category}")

    kind = (data_type or "").strip().lower()
    if kind == "review":
        return process_review_silver(df, category)
    if kind in ("metadata", "meta"):
        return process_meta_silver(df, category)

    raise ValueError("data_type must be 'review' or 'metadata'")

def clean_review_data(
    spark,
    category,
    target_parquet_file_mb=128,
    max_output_files=200,
    compression_codec="gzip",
):
    """
    Load review data from bronze zone, apply ETL transformations, 
    and write to silver zone.
    
    Parameters:
    -----------
    spark : pyspark.sql.SparkSession
        Spark session
    category : str
        Product category name
    target_parquet_file_mb : int
        Target file size in MB for output partitions
    max_output_files : int
        Maximum number of output files
    compression_codec : str
        Compression codec for output files
    """
    bronze_input_path = f"gs://amazon-reviews-lakehouse-data/bronze-zone/review-data/{category}"
    silver_output_path = f"gs://amazon-reviews-lakehouse-data/silver-zone/review-data/{category}"

    if _gcs_path_exists(silver_output_path, spark.sparkContext):
        print(f"Skip review category '{category}': output already exists at {silver_output_path}")
        return

    print(f"Processing review data for category: {category}")

    # Check if bronze data exists
    if not _gcs_path_exists(bronze_input_path, spark.sparkContext):
        print(f"Bronze zone data not found at {bronze_input_path}")
        return

    # Read from bronze zone
    print(f"Reading from bronze zone: {bronze_input_path}")
    df = spark.read.parquet(bronze_input_path)
    
    source_partitions = df.rdd.getNumPartitions()
    print(f"Source partition count: {source_partitions}")
    print(f"Source record count: {df.count()}")

    # Apply ETL transformations
    df_transformed = apply_etl_transformations(df, category, data_type="review")

    # Get input file size estimate for repartitioning
    # Read first parquet file to estimate size
    try:
        # This is a simple estimation; adjust based on actual use case
        num_files, parquet_size_bytes = _estimate_output_files_from_parquet_size(
            parquet_path=bronze_input_path,
            sc=spark.sparkContext,
            target_parquet_file_mb=target_parquet_file_mb,
            min_files=1,
            max_files=max_output_files,
        )
    except Exception:
        # Conservative fallback to avoid producing many tiny files.
        current_parts = df_transformed.rdd.getNumPartitions()
        num_files = max(1, min(max_output_files, max(1, current_parts // 4)))
        parquet_size_bytes = 0

    parquet_size_gb = parquet_size_bytes / (1024 ** 3) if 'parquet_size_bytes' in locals() else 0

    print(f"Estimated input parquet size: {parquet_size_gb:.2f} GB")
    print(f"Target parquet file size: {target_parquet_file_mb} MB")
    print(f"Calculated output files: {num_files}")
    print(f"Compression codec: {compression_codec}")

    # Repartition/coalesce for optimal file sizes
    current_partitions = source_partitions
    if num_files >= current_partitions:
        df_to_write = df_transformed.repartition(num_files)
        partition_action = "repartition"
    else:
        df_to_write = df_transformed.coalesce(num_files)
        partition_action = "coalesce"

    print(f"Input partitions: {current_partitions}, using {partition_action} -> {num_files}")

    # Write to silver zone
    (
        df_to_write
        .write
        .mode("overwrite")
        .option("compression", compression_codec)
        .parquet(silver_output_path)
    )

    print(f"Finished writing cleaned review data to: {silver_output_path}")

def clean_metadata(
    spark,
    category,
    target_parquet_file_mb=128,
    max_output_files=200,
    compression_codec="gzip",
):
    """
    Load metadata from bronze zone, apply ETL transformations, 
    and write to silver zone.
    
    Parameters:
    -----------
    spark : pyspark.sql.SparkSession
        Spark session
    category : str
        Product category name
    target_parquet_file_mb : int
        Target file size in MB for output partitions
    max_output_files : int
        Maximum number of output files
    compression_codec : str
        Compression codec for output files
    """
    bronze_input_path = f"gs://amazon-reviews-lakehouse-data/bronze-zone/meta-data/{category}"
    silver_output_path = f"gs://amazon-reviews-lakehouse-data/silver-zone/meta-data/{category}"

    if _gcs_path_exists(silver_output_path, spark.sparkContext):
        print(f"Skip metadata category '{category}': output already exists at {silver_output_path}")
        return

    print(f"Processing metadata for category: {category}")

    # Check if bronze data exists
    if not _gcs_path_exists(bronze_input_path, spark.sparkContext):
        print(f"Bronze zone data not found at {bronze_input_path}")
        return

    # Read from bronze zone
    print(f"Reading from bronze zone: {bronze_input_path}")
    df = spark.read.parquet(bronze_input_path)
    
    source_partitions = df.rdd.getNumPartitions()
    print(f"Source partition count: {source_partitions}")
    print(f"Source record count: {df.count()}")

    # Apply ETL transformations
    df_transformed = apply_etl_transformations(df, category, data_type="metadata")

    # Get input file size estimate for repartitioning
    try:
        num_files, parquet_size_bytes = _estimate_output_files_from_parquet_size(
            parquet_path=bronze_input_path,
            sc=spark.sparkContext,
            target_parquet_file_mb=target_parquet_file_mb,
            min_files=1,
            max_files=max_output_files,
        )
    except Exception:
        current_parts = df_transformed.rdd.getNumPartitions()
        num_files = max(1, min(max_output_files, max(1, current_parts // 4)))
        parquet_size_bytes = 0

    parquet_size_gb = parquet_size_bytes / (1024 ** 3) if 'parquet_size_bytes' in locals() else 0

    print(f"Estimated input parquet size: {parquet_size_gb:.2f} GB")
    print(f"Target parquet file size: {target_parquet_file_mb} MB")
    print(f"Calculated output files: {num_files}")
    print(f"Compression codec: {compression_codec}")

    # Repartition/coalesce for optimal file sizes
    current_partitions = source_partitions
    if num_files >= current_partitions:
        df_to_write = df_transformed.repartition(num_files)
        partition_action = "repartition"
    else:
        df_to_write = df_transformed.coalesce(num_files)
        partition_action = "coalesce"

    print(f"Input partitions: {current_partitions}, using {partition_action} -> {num_files}")

    # Write to silver zone
    (
        df_to_write
        .write
        .mode("overwrite")
        .option("compression", compression_codec)
        .parquet(silver_output_path)
    )

    print(f"Finished writing cleaned metadata to: {silver_output_path}")

In [6]:
spark, sc = create_spark_session(driver_memory_gb=24, executor_memory_gb=24, reserve_cores=2)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 10:31:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark cores: 2/4
Driver memory: 24g, Executor memory: 24g
defaultParallelism=8, shufflePartitions=16


Processing metadata for category: Books
Reading from bronze zone: gs://amazon-reviews-lakehouse-data/bronze-zone/meta-data/Books
Source partition count: 37


Source record count: 4448181
Applying ETL transformations for metadata category: Books
Estimated input parquet size: 4.37 GB
Target parquet file size: 128 MB
Calculated output files: 35
Compression codec: gzip
Input partitions: 37, using coalesce -> 35


Finished writing cleaned metadata to: gs://amazon-reviews-lakehouse-data/silver-zone-test/meta-data/Books


In [ ]:
# tam thoi bo Automotive
categories = [
"All_Beauty", "Amazon_Fashion", "Appliances", "Arts_Crafts_and_Sewing",
"Automotive","Books", "CDs_and_Vinyl", "Cell_Phones_and_Accessories",
"Clothing_Shoes_and_Jewelry", "Digital_Music", "Electronics", "Gift_Cards",
"Grocery_and_Gourmet_Food", "Health_and_Household", "Health_and_Personal_Care",
"Home_and_Kitchen", "Industrial_and_Scientific", "Kindle_Store",
"Magazine_Subscriptions", "Movies_and_TV", "Musical_Instruments",
"Office_Products", "Patio_Lawn_and_Garden", "Pet_Supplies", "Software",
"Sports_and_Outdoors", "Subscription_Boxes", "Tools_and_Home_Improvement",
"Toys_and_Games", "Video_Games", "Unknown"
]

for data_type in ["meta", "review"]:
    for category in categories:
        if data_type == "meta":
            clean_metadata(spark, category, target_parquet_file_mb=64, max_output_files=128, compression_codec="gzip")
        else:
            clean_review_data(spark, category, target_parquet_file_mb=64, max_output_files=128, compression_codec="gzip")